In [1]:
!pip install google-genai pydantic pillow datasets -q

In [1]:
# =====================================================================
# Top-level Step 1: Install and load all required packages
# =====================================================================

import os
import json
import random
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from datasets import load_dataset
from PIL import Image
from google.colab import userdata

# 🚀 Automatically retrieve the API key using Colab's built-in security mechanism
os.environ["GEMINI_API_KEY"] = userdata.get('gemini')
client = genai.Client()


# =====================================================================
# Top-level Step 2: Download the FUNSD dataset
# =====================================================================
print("⏳ Downloading the FUNSD dataset from Hugging Face...")
dataset = load_dataset("davidle7/funsd-json")
print("✅ Dataset loaded successfully!")

⏳ Downloading the FUNSD dataset from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.39k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/26.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/9.72M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/149 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50 [00:00<?, ? examples/s]

✅ Dataset loaded successfully!


In [ ]:
'''
sample_index = 23
sample_data = dataset['train'][sample_index]
funsd_ground_truth = json.loads(sample_data['text_output'])
funsd_ground_truth
'''

In [ ]:
import time
import json
from pydantic import BaseModel, Field

# =====================================================================
# Top-level Step 3: Define data structures (Structured Outputs Schema)
# =====================================================================
class KeyValuePair(BaseModel):
    key: str = Field(description="Field name (question) in the form, e.g., Name, Date, Registration No.")
    value: str = Field(description="Specific content filled in the field (answer), e.g., John, 11/14/98, 533")

class ExtractedForm(BaseModel):
    pairs: list[KeyValuePair] = Field(description="List of all successfully paired key-value pairs in this form")


# =====================================================================
# Top-level Step 4: Define the Python Grader (Scoring Function)
# =====================================================================
def evaluate_predictions(worker_output: dict, ground_truth: dict) -> tuple[float, str]:
    """
    Compare the JSON generated by the Worker with the newly custom ground truth,
    calculate the matching accuracy based on exact key-value string overlap,
    and generate an error log for the Judge model.
    """
    # 1. Parse your newly custom, clean ground truth into a standardized lowercase set
    gt_pairs = set()
    for pair in ground_truth.get("pairs", []):
        k = pair.get("key", "").strip().lower()
        v = pair.get("value", "").strip().lower()
        if k and v:
            gt_pairs.add((k, v))

    # 2. Parse the pairs extracted by the Worker model (Gemini Flash)
    worker_pairs = set()
    for pair in worker_output.get("pairs", []):
        k = pair.get("key", "").strip().lower()
        v = pair.get("value", "").strip().lower()
        if k and v:
            worker_pairs.add((k, v))

    # Handle edge case where ground truth dataset might be empty
    if not gt_pairs:
        return 100.0, "No available annotations for this form"

    # 3. Calculate metrics using standard set intersections
    correct_matches = worker_pairs.intersection(gt_pairs)
    missing_matches = gt_pairs - worker_pairs

    # Score represents the percentage of correctly retrieved golden pairs
    score = (len(correct_matches) / len(gt_pairs)) * 100

    # 4. Generate the structured error log for the Judge optimization step
    error_log = f"Current Score: {score:.1f}%\n"
    error_log += f"❌ Examples of missed or incorrectly recognized key fields:\n"

    if missing_matches:
        for i, (missing_k, missing_v) in enumerate(list(missing_matches)[:3]):
            error_log += f"   - Should have captured: '{missing_k}' -> '{missing_v}', but the model missed it or paired it incorrectly.\n"
    else:
        error_log += "   None! All fields matched perfectly.\n"

    return score, error_log


# =====================================================================
# Top-level Step 5: Main Program - Auto-Optimization Loop (Optimization Loop)
# =====================================================================

# A. Lock onto the designated document image
sample_index = 23
sample_data = dataset['train'][sample_index]
funsd_image = sample_data['image']

# 🌟 UPDATED: Injecting the clean, human-intuitive Ground Truth verified by the Pro model
funsd_ground_truth = {
  "pairs": [
    {"key": "Date", "value": "September 25, 1969"},
    {"key": "Agency", "value": "SSC&B, Inc"},
    {"key": "Contact", "value": "Larry Odum"},
    {"key": "Address", "value": "575 Lexington Avenue, New York, New York 10022"},
    {"key": "Client", "value": "American Tobacco Company"},
    {"key": "Product", "value": "Pall Mall Filter tip"},
    {"key": "Film Cleared", "value": "VIDEO TAPE APPROVAL"},
    {"key": "Independent Stamp / Status", "value": "RECEIVED SEP 30 1969 PERCY F. SMITH"},
    {"key": "CF", "value": "cb"}
  ]
}

# B. Set the initial (1st generation) enhanced prompt with the General Inline List Rule
current_prompt = """You are a professional form OCR recognition expert. Please adhere to the following strict rules to extract Key-Value pairs:

1. [Standard Fields]: Pair the nearest and aligned question (Key) with its answer (Value).

2. [Inline Lists & Table Rows with Special Characters]: If you encounter a horizontal list or a table-like row where multiple values are aligned on a single line (often separated by spaces, dashes, or special characters like colons ":", slashes "/", or hashtags "#"), do NOT strictly split the key-value pair at the first special character. Instead, treat the leftmost descriptive text/name as the "key", and capture the entire sequence of codes, numbers, or identifiers on the right as the integrated "value". Do not drop the numeric identifiers or metrics at the end of the line.

3. [Independent Stamps & Status]: Please look out for "receipt/received stamps (e.g., RECEIVED)" or "approval status (e.g., VIDEO TAPE APPROVAL)" at the edges of the image; these are usually independent Key-Value pairs as well.

4. Ensure the text matches the image perfectly; do not hallucinate layout or content."""

# History logging initialization
prompt_history = []
json_history = {}

print(f"\n🎯 Locking onto file #{sample_index} in the FUNSD training set for Prompt evolution!")
max_iterations = 3  # Maximum evaluation iterations allowed

for iteration in range(1, max_iterations + 1):
    # Log the prompt configuration used at the beginning of this generation step
    prompt_history.append({"generation": iteration, "prompt": current_prompt})

    print(f"\n==================================================")
    print(f"🔄 【Generation {iteration} Prompt Iteration Started】")
    print(f"==================================================")
    print(f"💡 Current instruction (Prompt) used:\n\"{current_prompt}\"\n")

    # ─── Task A: Worker attempts the visual recognition task ───
    print("🤖 Calling Gemini 2.5 Flash for image recognition...")
    worker_response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[funsd_image, current_prompt],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ExtractedForm, # Constrain API schema format
            temperature=0.1 # Maintain low temperature for extraction deterministic consistency
        ),
    )

    try:
        worker_json = json.loads(worker_response.text)
    except Exception as e:
        print(f"💥 JSON parsing error: {e}")
        worker_json = {"pairs": []}

    # ─── Task B: Python program evaluates the extracted JSON ───
    score, error_report = evaluate_predictions(worker_json, funsd_ground_truth)
    print(f"\n📊 Grading Result: Received {score:.1f} points (Target: 85.0 points or above)")

    # Cache execution metadata immediately into log
    json_history[iteration] = {
        "score": score,
        "json_data": worker_json
    }

    # Early exit condition if accuracy metric passes target
    if score >= 85.0:
        print("🎉 Congratulations! The score has reached the target. Evolution complete! This is the final golden Prompt.")
        break

    if iteration == max_iterations:
        print("🛑 Maximum iterations reached. Evolution complete.")
        break

    # ─── Task C: Judge triggers prompt mutation upon suboptimal score ───
    print("\n🧠 Score below target, calling Gemini 2.5 Flash to optimize the prompt...")

    judge_system_instruction = """
    You are an AI Prompt Optimization Master. Your task is to analyze the reasons for errors made by the Worker model when extracting key-value pairs from paper forms, and to correct and reinforce the [Original Prompt].

    Please include specific "strategic guidance" regarding structure, spatial layout, alignment, or font treatment in the modified prompt (e.g., pay attention to horizontal alignment, pay attention to upper and lower adjacent cells).

    ⚠️ Constraints: Please directly output only the "optimized brand-new prompt content" itself. Do not include any introduction, quotation marks, or noise like "Sure, here is the modified prompt."
    """

    judge_user_content = f"""
    【Original Prompt】:
    "{current_prompt}"

    【Error Report and Missed Fields Provided by the Teacher】:
    {error_report}

    Please help me rewrite this prompt and add clear constraints so that the Worker won't make the same mistakes when encountering similar layouts again.
    """

    judge_response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=judge_user_content,
        config=types.GenerateContentConfig(
            system_instruction=judge_system_instruction,
            temperature=0.2  # Keep judge stable but adaptive
        )
    )

    # Overwrite Prompt state variable to prepare for next generation execution step
    current_prompt = judge_response.text.strip()

    # 🌟 ADDED: Rate-limiting safety pause to strictly comply with Google Free Tier Limits (15 RPM)
    if iteration < max_iterations:
        print("⏳ Cooling down for 6 seconds to respect API Free Tier Rate Limits...")
        time.sleep(6)

print(f"\n🏁 The final golden Prompt generated by evolution is:\n\"{current_prompt}\"")

# Print out historical prompt transformations sequentially
print("\n📜 ==================================================")
print("📜           --- PROMPT EVOLUTION HISTORY LOG ---")
print("\n📜 ==================================================")
for history in prompt_history:
    print(f"【Generation {history['generation']} Prompt】:")
    print(f"\"{history['prompt']}\"")
    print("-" * 50)

In [ ]:
# =====================================================================
# View JSON answer comparison across all generated iterations (Loops through Gen 1 to Max)
# =====================================================================

print("==================================================")
print(" 📊      --- ALL GENERATED JSON ANSWERS ---        ")
print("==================================================")

# Check if history has any records to display
if not json_history:
    print("❌ No records found in json_history. Please run the evolution loop first.")
else:
    # Dynamically loop through all recorded iterations in sorted order (e.g., 1, 2, 3)
    for gen_num in sorted(json_history.keys()):
        score = json_history[gen_num]['score']
        json_data = json_history[gen_num]['json_data']

        print(f"\n▶️【JSON Answer Generated by Gen {gen_num} Prompt】")
        print(f"   Final Verification Score: {score:.1f}%")
        print("--------------------------------------------------")

        # indent=2 automatically formats and indents JSON to make it neat and human-readable
        print(json.dumps(json_data, indent=2, ensure_ascii=False))
        print("==================================================")

In [ ]:
import json
from PIL import Image

# 1. Fetch file number 23
target_index = 23
exam_data = dataset['train'][target_index]

# 2. Obtain image and official ground truth
exam_image = exam_data['image']
exam_gt = json.loads(exam_data['text_output'])

# 3. Save the image to Colab's left folder
exam_image.save("/content/exam_23.jpg")
print("✅ Image successfully saved as 'exam_23.jpg'!")

# 4. Display this image directly inside Colab
print("\n👀 This is the image for exam/sample #23:")
display(exam_image)

# 5. (Optional) Print what the ground truth looks like for easy comparison
print("\n📜 Official Ground Truth (First 5 elements):")
for element in exam_gt.get("form", [])[:5]:
    print(f"ID {element['id']}: [{element['label']}] -> '{element['text']}'")